# DS2002 · Mini-ETL on Walmart Sample

**Studio — 2026-10-14 · Fall 2026**  
**Class time:** 45 minutes

---

## The midterm, in miniature

Twelve rows of the real thing. Every problem in this sample is in the full dataset you get Monday, just at a scale where you can still see the answer.

You are writing the whole pipeline as **functions**, not as loose cells. That is not stylistic — Monday you will run the same logic on a much larger file, and a pipeline built from functions is one you can re-run and hand to a teammate. A pipeline built from cells in a specific click order is not.

The extract stage is worked below as an example. Transform and load are yours to build.

In [ ]:
import pandas as pd, sqlite3
from io import StringIO

RAW = '''transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1,3.86
TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1,5.00
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3,$4.11
TXN5,fl-239,2024-09-06T12:15:00,WATER-24,bottled water,grocery,4,$5.00
TXN6,FL-105,2024-09-07 08:02:00,BTL-WATER-24,Bottled Water 24pk,Grocery,6,5.00
TXN7,FL-105,2024-09-07T09:45:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,5,4.11
TXN8,FL-239,,PT-12,pop-tarts strawberry,Breakfast,2,$4.11
TXN9,FL-105,2024-09-07 14:20:00,FLASH-AA,Flashlight AA,Hardware,NULL,12.99
TXN10,FL-239,2024-09-07T15:00:00,POPTART-BLUE,Pop-Tarts Blueberry,Breakfast,1,4.11
TXN11,FL-105,09/07/2024 16:10,SKU#459812,Strawberry Pop-Tart,breakfast,4,4.11'''

### Extract — worked example

The extract function does one job: get the raw data in, and report what arrived. It does **not** clean anything. Keeping the stages separate is what lets you answer "was this a source problem or did I do it?" later.

In [ ]:
def extract():
    """Read the raw transactions. No cleaning here -- report only."""
    df = pd.read_csv(StringIO(RAW))
    print(f'extracted {len(df)} rows, {len(df.columns)} columns')
    print('dtypes that arrived as text:',
          [c for c in df.columns if df[c].dtype == object])
    print('nulls:', df.isnull().sum().sum())
    print('exact duplicates:', df.duplicated().sum())
    return df

raw = extract()
raw.head()

Note what the report already told us: `quantity` and `unit_price` both arrived as text, there is a null, and there is a duplicate. That is your transform to-do list, generated from the data instead of from guessing.

**TODO:** before writing any transform, list the specific problems you can see in these twelve rows. There are at least six.

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._
6. _..._

### Build 1 — Transform: types and duplicates

**TODO:** write `transform_types(df)` that returns a new frame with:

- exact duplicate rows removed
- `unit_price` as a float
- `quantity` numeric, with a stated decision about the null and the negative
- `store_id` normalized — note the four spellings of the same two stores
- `timestamp` as a real datetime, coercing the empty one

It must not modify the frame passed in. Print a before/after row count.

In [ ]:
def transform_types(df):
    out = df.copy()
    # TODO: dedupe
    # TODO: unit_price -> float
    # TODO: quantity -> numeric, then your decision on null / negative
    # TODO: store_id -> one consistent format
    # TODO: timestamp -> datetime, errors='coerce', format='mixed'
    return out

typed = transform_types(raw)
print('rows:', len(raw), '->', len(typed))
typed.dtypes

### Build 2 — Transform: consolidate the products

Four SKUs are strawberry Pop-Tarts. One is blueberry — leave it alone, and be ready to say why. Two are the same bottled water.

**TODO:** write `consolidate(df)` that adds a `canonical_sku` column. Any SKU not in your mapping should keep its original value, not become `NaN`.

Then print units per canonical SKU and confirm the total quantity did not change — consolidation renames things, it never adds or removes units.

In [ ]:
CANON = {
    # TODO: map the strawberry Pop-Tart SKUs to one value
    # TODO: map the two bottled water SKUs to one value
}

def consolidate(df):
    out = df.copy()
    # TODO: out['canonical_sku'] = ...  (keep unmapped SKUs as themselves)
    return out

consolidated = consolidate(typed)
# TODO: assert total quantity is unchanged from `typed`
# TODO: print units per canonical_sku

**Why blueberry stays separate:** _..._

### Build 3 — Transform: derive what the analysis needs

**TODO:** add `revenue` (quantity times unit price) and a `date` column from the timestamp. The `date` column is what you will join weather onto next week, so it needs to be a date — not a full timestamp, not a string.

In [ ]:
def derive(df):
    out = df.copy()
    # TODO: revenue
    # TODO: date (hint: .dt.date, or .dt.normalize() to keep it a datetime)
    return out

enriched = derive(consolidated)
enriched[['transaction_id', 'canonical_sku', 'quantity', 'revenue', 'date']]

### Build 4 — Load, and verify the load

**TODO:** write `load(df, conn)` that writes the clean frame to a `clean_sales` table, then reads back the row count and asserts it matches. Writing without verifying is how you discover at 11pm that `if_exists='append'` doubled your table.

In [ ]:
def load(df, conn, table='clean_sales'):
    # TODO: df.to_sql(...) with index=False and if_exists='replace'
    # TODO: read back COUNT(*) and assert it equals len(df)
    # TODO: print confirmation
    pass

conn = sqlite3.connect(':memory:')
load(enriched, conn)

### Build 5 — run the whole thing as one pipeline

**TODO:** wire the stages together in one function, then run it twice against a fresh connection and confirm you get identical results. That is the idempotency property from Monday, tested rather than assumed.

In [ ]:
def run_pipeline(conn):
    # TODO: extract -> transform_types -> consolidate -> derive -> load
    # TODO: return the final frame
    pass

first = run_pipeline(sqlite3.connect(':memory:'))
second = run_pipeline(sqlite3.connect(':memory:'))
# TODO: assert the two runs produced the same shape and the same revenue total

### Build 6 — the question the pipeline exists to answer

**TODO:** did strawberry Pop-Tart sales rise from Sept 6 to Sept 7? Print units by date for your canonical strawberry SKU, and say whether two days is enough evidence to conclude anything.

That second part matters. On the midterm you will have a real baseline to compare against; here you do not, and saying so is the correct answer.

In [ ]:
# TODO

---

## Checkpoint (participation)

Report your row count after transform, your canonical SKU count, and whether running the pipeline twice gave the same revenue total.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [ ]:
# Checkpoint
rows_after_transform = None   # TODO
canonical_skus = None         # TODO: how many distinct products you ended with
idempotent = None             # TODO: True/False from Build 5
blueberry_call = 'TODO'       # TODO: why you kept blueberry separate

print('rows after transform:', rows_after_transform)
print('canonical SKUs:', canonical_skus)
print('same result twice:', idempotent)
print('blueberry:', blueberry_call)